In [29]:
import os

from dotenv import load_dotenv
load_dotenv()


api_key = os.getenv("OPENAI_API_KEY")


In [26]:
from openai import OpenAI

openai_client = OpenAI(api_key=api_key)


In [27]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    
    return response.output_text


In [28]:
question = "Hello"
answer = llm(question)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)


In [ ]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'

response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [ ]:
documents = []

url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    
    course_url = f"""{url_prefix}{course['path']}"""
    
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    
    documents.extend(course_data)
    
    
len(documents)

1400

In [ ]:
documents[3]

{'id': 'b5d37f88b6',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How are homework assignments scored?',
 'answer': "Homework grade = points for the questions + 1 point for the FAQ + up to 7 points for learning in public.\n\n- Learning in public: up to 7 points, depending on how many platforms you shared your post on (e.g. LinkedIn, X, blog) and the quality of the post.\n- FAQ: 1 point. To get it, contribute to the FAQ repo and add the link to your PR in your homework submission.\n\nOptional questions are scored too - read the submission form carefully. The homework is for practice and does not affect your certificate. If you think you weren't graded, log in to check your results or search for your name on the leaderboard."}

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)


In [ ]:
question = "I just discovered the course. Can I join now"

search_results = index.search(
    question,
    boost_dict={
        'question': 2,
        'section': 1,
        'answer': 1,
    },
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [ ]:
def search(question, course="llm-zoomcamp"):
    
    boost_dict = {
        'question': 2,
        'section': 1,
        'answer': 1,
    }
    filter_dict = {"course": course}
    
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    
    lines = []
    
    for doc in search_results:
        lines.append(f"Section: {doc['section']}")
        lines.append(f"Question: {doc['question']}")
        lines.append(f"Answer: {doc['answer']}")
        lines.append("\n")
    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    
    context = build_context(search_results)
    
    user_prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    
    return f"{INSTRUCTIONS}\n{user_prompt}"

In [ ]:
question = "I just discovered the course. Can I join now"

search_results = search(question)

prompt = build_prompt(question, search_results)
prompt

'\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\n\nQuestion:\nI just discovered the course. Can I join now\n\nContext:\nSection: General Course-Related Questions\nQuestion: I just discovered the course. Can I still join?\nAnswer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\n\nSection: General Course-Related Questions\nQuestion: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nAnswer: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\n\nSection: General Course

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)